# Error Visibility

## What you'll learn

- How the pipeline handles empty filter results without crashing
- How to diagnose skipped steps using `inspect_pipeline` and step metadata
- How to use `passthrough_failures` for soft quality gating

**Prerequisites:** [First Pipeline](../getting-started/01-first-pipeline.ipynb), [Metrics and Filtering](03-metrics-and-filtering.ipynb)  
**Estimated time:** 10 minutes  
**GPU required:** No.

---

In production pipelines, filters sometimes eliminate all candidates. A quality
threshold might be too strict, or upstream data didn't produce anything viable.
When this happens, the pipeline doesn't crash. It detects empty inputs, skips
downstream steps, and records what happened so you can diagnose the issue.

This tutorial walks through that behavior and shows you how to inspect and
control it.

| Section | What you'll see |
|---------|----------------|
| **The empty filter cascade** | A strict filter passes zero artifacts; downstream steps skip |
| **Diagnosing skipped steps** | Using `inspect_pipeline` and step metadata to find where artifacts were lost |
| **Soft gating** | `passthrough_failures` evaluates criteria but passes all artifacts through |

In [ ]:
from __future__ import annotations

from artisan.operations.curator import Filter
from artisan.operations.examples import (
    DataGeneratorWithMetrics,
    DataTransformer,
    MetricCalculator,
)
from artisan.orchestration import PipelineManager
from artisan.utils import tutorial_setup
from artisan.utils.logging import configure_logging
from artisan.visualization import build_macro_graph, build_micro_graph, inspect_pipeline

configure_logging()

In [ ]:
env = tutorial_setup("error_visibility")

## The empty filter cascade

`DataGeneratorWithMetrics` produces 5 datasets with `mean_score` values in
\[0, 1\]. We'll filter with a threshold of `> 100` — impossible to satisfy.
Nothing passes, and every downstream step is skipped automatically.

In [ ]:
pipeline = PipelineManager.create(
    name="empty_filter_cascade",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

# Step 0: Generate 5 datasets with co-produced metrics
step_0 = pipeline.run(
    operation=DataGeneratorWithMetrics, name="generate", params={"count": 5, "seed": 42}
)

# Step 1: Filter with impossible threshold — nothing passes
step_1 = pipeline.run(
    operation=Filter,
    name="filter",
    inputs={"passthrough": output("generate", "datasets")},
    params={
        "criteria": [{"metric": "mean_score", "operator": "gt", "value": 100}],
    },
)

# Step 2: Receives empty inputs — will be skipped
step_2 = pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("filter", "passthrough")},
    params={"seed": 100},
)

# Step 3: Cascade continues — also skipped
step_3 = pipeline.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("transform", "dataset")},
)

result = pipeline.finalize()
print(f"Pipeline complete: success={result['overall_success']}")

The pipeline finished without error. Use `inspect_pipeline` to see what
happened at each step:

In [ ]:
inspect_pipeline(env.delta_root)

Steps 0 and 1 completed — step 1 evaluated the filter and passed zero
artifacts. Steps 2 and 3 show `status = "skipped"` with no produced
artifacts and no duration. They never executed.

In [ ]:
build_macro_graph(env.delta_root)

In [ ]:
build_micro_graph(env.delta_root)

Notice the macro graph only shows steps 0 and 1. Skipped steps are excluded
from the graph entirely. If your graph looks shorter than expected, check
`inspect_pipeline` — that's where skipped steps appear.

## Diagnosing skipped steps

Each call to `pipeline.run()` returns a `StepResult`. When a step is
skipped, its `metadata` dict tells you why:

| Skip reason | Meaning |
|------------|--------|
| `"empty_inputs"` | This step was the first to receive zero artifacts |
| `"pipeline_stopped"` | This step was skipped because an earlier step already triggered the cascade |

The distinction tells you where to look: the `"empty_inputs"` step is
where artifacts were lost. Trace back from there to find the filter or
upstream step that produced nothing.

In [ ]:
for step in [step_0, step_1, step_2, step_3]:
    skipped = step.metadata.get("skipped", False)
    if skipped:
        reason = step.metadata["skip_reason"]
        print(f"Step {step.step_number} ({step.step_name}): skipped — {reason}")
    else:
        print(
            f"Step {step.step_number} ({step.step_name}): {step.succeeded_count} succeeded"
        )

Step 2 (`transform`) was the first to detect empty inputs —
that's where the cascade started. Step 3 (`metrics`) was
skipped because the pipeline had already stopped.

:::{note}
**Skipped vs. cached:** These are different concepts. A *cached* step
skips execution because its results already exist from a previous run.
A *skipped* step had nothing to process. Skipped steps are **not**
cached — if you rerun the pipeline with different inputs, they will
re-evaluate.
:::

## Soft gating with `passthrough_failures`

Sometimes you want to evaluate filter criteria without actually removing
artifacts. Setting `passthrough_failures=True` makes Filter act as a
diagnostic tool: it records which artifacts would fail, but passes
everything through.

This is useful when you want to:
- Monitor quality trends without blocking the pipeline
- Test a new filter criterion before making it strict
- Collect pass/fail rates without enforcement

We'll run the same pipeline with the same impossible threshold, but this
time with `passthrough_failures=True`.

In [ ]:
env_soft = tutorial_setup("error_visibility_soft")

In [ ]:
pipeline_soft = PipelineManager.create(
    name="soft_gating",
    delta_root=env_soft.delta_root,
    staging_root=env_soft.staging_root,
    working_root=env_soft.working_root,
)
output = pipeline_soft.output

step_0s = pipeline_soft.run(
    operation=DataGeneratorWithMetrics, name="generate", params={"count": 5, "seed": 42}
)

# Same impossible threshold, but passthrough_failures=True
step_1s = pipeline_soft.run(
    operation=Filter,
    name="filter",
    inputs={"passthrough": output("generate", "datasets")},
    params={
        "criteria": [{"metric": "mean_score", "operator": "gt", "value": 100}],
        "passthrough_failures": True,
    },
)

step_2s = pipeline_soft.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("filter", "passthrough")},
    params={"seed": 100},
)

step_3s = pipeline_soft.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("transform", "dataset")},
)

result_soft = pipeline_soft.finalize()
print(f"Pipeline complete: success={result_soft['overall_success']}")

In [ ]:
inspect_pipeline(env_soft.delta_root)

All four steps completed. Filter still evaluated the criteria — zero
artifacts passed the `> 100` threshold — but with `passthrough_failures=True`
it output all 5 artifacts regardless. Downstream steps ran normally.

The filter diagnostics are still recorded in the step metadata, so you
can inspect which artifacts *would* have been eliminated. Compare with
the first pipeline where the same threshold caused steps 2–3 to skip.

In [ ]:
build_macro_graph(env_soft.delta_root)

In [ ]:
build_micro_graph(env_soft.delta_root)

The macro graph now shows all four steps — nothing was skipped.

## Summary

This tutorial covered how pipelines handle empty filter results and how
you control that behavior.

| Scenario | Behavior | Use when |
|----------|----------|----------|
| Strict filter (default) | Downstream steps skip; recorded as `"skipped"` | Hard quality gates — only passing artifacts continue |
| `passthrough_failures=True` | All artifacts pass; diagnostics still recorded | Monitoring — log pass/fail without enforcement |

Key points:

- The pipeline never crashes on empty filter results — downstream steps
  skip gracefully
- `inspect_pipeline` shows step statuses including skipped steps;
  `build_macro_graph` only shows completed steps
- Check `step.metadata["skip_reason"]` to find where artifacts were
  lost: `"empty_inputs"` marks the origin, `"pipeline_stopped"` marks
  the cascade
- Skipped steps are not cached — they re-evaluate if inputs change on
  a subsequent run

## Next steps

- [Working with Results](../working-with-results/01-provenance-graphs.ipynb) — macro and micro provenance graph rendering
- [Error Handling](../../concepts/error-handling.md) — design rationale for how the framework contains and reports failures